<a href="https://colab.research.google.com/github/juanfebotero/juanfebotero.github.io/blob/main/notebooks/tecnicas-optimizacion/01_programacion_lineal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Programación lineal con PuLP: formular, resolver y visualizar

**Técnicas de Optimización** · Prof. Juan Felipe Botero · Universidad de Antioquia

**Objetivos**

- Formular un problema de programación lineal (PL) a partir de un enunciado.
- Resolverlo gráficamente y con el solver CBC a través de PuLP.
- Interpretar precios sombra y holguras.
- Ver qué cambia al exigir variables enteras.

> **Antes de empezar:** use *Archivo → Guardar una copia en Drive*. Si el cuaderno le llegó por Google Classroom, ya es su copia personal y el profesor verá su progreso.

In [ ]:
!pip -q install pulp
import pulp, numpy as np, matplotlib.pyplot as plt

## 1. El problema

Un operador de internet en Medellín vende dos planes:

| Plan | Ganancia mensual | Ancho de banda reservado | Horas de instalación |
|---|---|---|---|
| Residencial ($x_1$) | 40 mil COP | 10 Mbps | 2 h |
| Empresarial ($x_2$) | 90 mil COP | 50 Mbps | 3 h |

Este mes dispone de **1000 Mbps** en el enlace de agregación, **120 horas** de técnicos y solo **50 puertos** residenciales libres.

$$\max\; z = 40x_1 + 90x_2$$
$$\text{s.a.}\quad 10x_1 + 50x_2 \le 1000,\quad 2x_1 + 3x_2 \le 120,\quad x_1 \le 50,\quad x_1,x_2 \ge 0$$

## 2. Solución gráfica
Con dos variables podemos dibujar la región factible. El óptimo de un PL siempre está en un vértice.

In [ ]:
x = np.linspace(0, 70, 400)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(x, (1000 - 10*x)/50, label="ancho de banda")
ax.plot(x, (120 - 2*x)/3, label="horas técnicos")
ax.axvline(50, color="gray", ls="--", label="puertos")
X, Y = np.meshgrid(np.linspace(0, 70, 300), np.linspace(0, 45, 300))
fact = (10*X + 50*Y <= 1000) & (2*X + 3*Y <= 120) & (X <= 50)
ax.contourf(X, Y, fact, levels=[0.5, 1], alpha=.25)
for z in [1500, 2500, 3300]:
    ax.plot(x, (z - 40*x)/90, ":", color="k", lw=.8)
ax.set(xlim=(0, 70), ylim=(0, 45), xlabel="$x_1$ residencial", ylabel="$x_2$ empresarial")
ax.legend(); plt.show()

Las líneas punteadas son curvas de nivel de $z$. Al desplazarlas hacia arriba y a la derecha, la última que toca la región factible marca el óptimo.

## 3. Resolver con PuLP

In [ ]:
modelo = pulp.LpProblem("planes_isp", pulp.LpMaximize)
x1 = pulp.LpVariable("x1_residencial", lowBound=0)
x2 = pulp.LpVariable("x2_empresarial", lowBound=0)

modelo += 40*x1 + 90*x2, "ganancia"
modelo += 10*x1 + 50*x2 <= 1000, "ancho_banda"
modelo += 2*x1 + 3*x2 <= 120, "horas"
modelo += x1 <= 50, "puertos"

modelo.solve(pulp.PULP_CBC_CMD(msg=0))
print("Estado:", pulp.LpStatus[modelo.status])
print("x1 =", x1.value(), " x2 =", x2.value(), " z =", pulp.value(modelo.objective))

## 4. Precios sombra y holguras
El **precio sombra** de una restricción dice cuánto mejora $z$ si aumentamos su lado derecho en una unidad. Una restricción con holgura positiva no está activa y su precio sombra es cero.

In [ ]:
for nombre, c in modelo.constraints.items():
    print(f"{nombre:12s} precio sombra = {c.pi:6.2f}   holgura = {c.slack:6.2f}")

**Pregunta para pensar:** ¿vale la pena pagar 1 mil COP por hora extra de técnico? ¿Y 1 mil COP por Mbps adicional?

## 5. Ejercicios
**Ejercicio 1.** El operador consigue 1200 Mbps en lugar de 1000. Modifique el modelo, resuélvalo y guarde la nueva ganancia óptima en `resp_1`.

In [ ]:
# Escriba aquí su solución
resp_1 = None

**Ejercicio 2.** No se pueden vender fracciones de plan. Resuelva el modelo original con variables **enteras** (`cat='Integer'`) y guarde la ganancia en `resp_2`. ¿Por qué cambia (o no) respecto al PL?

In [ ]:
# Escriba aquí su solución
resp_2 = None

**Ejercicio 3. Asignación.** Cuatro técnicos deben atender cuatro zonas; la matriz `costo[i][j]` es el tiempo (h) del técnico *i* en la zona *j*. Formule el problema de asignación con variables binarias $y_{ij}$ (cada técnico una zona, cada zona un técnico) y guarde el tiempo mínimo total en `resp_3`.

In [ ]:
costo = [[9, 2, 7, 8], [6, 4, 3, 7], [5, 8, 1, 8], [7, 6, 9, 4]]
# Escriba aquí su solución
resp_3 = None

In [ ]:
# --- Autoevaluación: no modifique esta celda ---
import math
def verificar(nombre, valor, esperado, tol=1e-6):
    """Compara su respuesta con la esperada y muestra el resultado."""
    if valor is None:
        print(f"[ ] {nombre}: pendiente, complete la celda anterior.")
        return False
    try:
        ok = abs(float(valor) - float(esperado)) <= tol * max(1, abs(float(esperado)))
    except (TypeError, ValueError):
        ok = valor == esperado
    print(f"[{'✔' if ok else '✘'}] {nombre}: {'correcto' if ok else 'revise su respuesta (obtuvo ' + repr(valor) + ')'}")
    return ok

verificar('Ejercicio 1', resp_1, 2914.2856899999997)
verificar('Ejercicio 2', resp_2, 2710.0)
verificar('Ejercicio 3', resp_3, 13)

## Para ir más lejos
- Documentación de PuLP: https://coin-or.github.io/pulp/
- Pruebe cambiar el solver o escribir el modelo en forma matricial con `numpy` y `scipy.optimize.linprog`.